# JARVIS-DFT Thermoelectric Data -> SISSO-Ready Dataset
Bismuth-telluride + Half-Heusler families, with the agreed variable set:

**Target (y):** Seebeck coefficient (n- and p-type combined via long format)

**Tier 1 input variables (X):**
- carrier_sign (-1 = n-type, +1 = p-type)
- mean electronegativity
- mean valence electron count
- mean covalent radius
- band gap (optb88vdw_bandgap)
- electronegativity range

**Tier 2 (optional, added but easy to drop):**
- mean first ionization energy
- mean melting point
- mean unfilled orbitals
- ehull (stability, filter/sanity use)

Seebeck reference point in JARVIS: 600 K, 10^20 cm^-3 doping. n-type negative, p-type positive.

Run top to bottom in Google Colab.

## 1. Install dependencies

In [ ]:
!pip install jarvis-tools pandas matminer pymatgen -q

## 2. Download JARVIS dft_3d

In [ ]:
import pandas as pd
from jarvis.db.figshare import data as jdata

d = jdata("dft_3d")
df = pd.DataFrame(d)
print(f"Total entries: {len(df)}")

## 3. Elements + TE-data flag

In [ ]:
def get_elements(row):
    try:
        return set(row["atoms"]["elements"])
    except Exception:
        return set()

df["elements"] = df.apply(get_elements, axis=1)
df["n_elements"] = df["elements"].apply(len)

def has_te_data(row):
    return (row.get("n-Seebeck", "na") not in ("na", None)) or (row.get("p-Seebeck", "na") not in ("na", None))

df["has_te_data"] = df.apply(has_te_data, axis=1)

keep_cols = ["jid", "formula", "spg_number", "n-Seebeck", "p-Seebeck", "optb88vdw_bandgap", "ehull"]
keep_cols = [c for c in keep_cols if c in df.columns]

## 4. Filter into the two families

In [ ]:
pnictogens = {"Bi", "Sb"}
chalcogens = {"Te", "Se", "S"}

def is_bite_family(elems):
    return bool(elems & pnictogens) and bool(elems & chalcogens)

bite_mask = df["elements"].apply(is_bite_family) & df["has_te_data"]
df_bite = df.loc[bite_mask, keep_cols].reset_index(drop=True)
df_bite["family"] = "bismuth_telluride"

def is_half_heusler(row):
    if row["n_elements"] != 3:
        return False
    try:
        if int(row.get("spg_number", -1)) != 216:
            return False
    except (ValueError, TypeError):
        return False
    comp = row["atoms"]["elements"]
    counts = {e: comp.count(e) for e in set(comp)}
    vals = list(counts.values())
    return len(vals) == 3 and max(vals) == min(vals)

hh_mask = df.apply(is_half_heusler, axis=1) & df["has_te_data"]
df_hh = df.loc[hh_mask, keep_cols].reset_index(drop=True)
df_hh["family"] = "half_heusler"

df_combined = pd.concat([df_bite, df_hh], ignore_index=True)
print(f"Bi-Te family: {len(df_bite)} | Half-Heusler: {len(df_hh)} | Combined: {len(df_combined)}")

## 5. Reshape to long format (n/p -> carrier_sign)

In [ ]:
df_long = pd.melt(
    df_combined,
    id_vars=[c for c in df_combined.columns if c not in ["n-Seebeck", "p-Seebeck"]],
    value_vars=["n-Seebeck", "p-Seebeck"],
    var_name="doping_type",
    value_name="Seebeck"
)
df_long["doping_type"] = df_long["doping_type"].str.replace("-Seebeck", "")
df_long = df_long[df_long["Seebeck"] != "na"].copy()
df_long["Seebeck"] = df_long["Seebeck"].astype(float)
df_long["carrier_sign"] = df_long["doping_type"].map({"n": -1, "p": 1})

print(f"Long-format entries: {len(df_long)}")

## 6. Compute Magpie elemental descriptors, keep only the agreed variables

In [ ]:
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition

featurizer = ElementProperty.from_preset("magpie")
feature_labels = featurizer.feature_labels()

def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

df_long["composition"] = df_long["formula"].apply(safe_composition)
df_long = df_long[df_long["composition"].notna()].reset_index(drop=True)

def featurize_row(comp):
    try:
        return featurizer.featurize(comp)
    except Exception:
        return [None] * len(feature_labels)

features = df_long["composition"].apply(featurize_row)
features_df = pd.DataFrame(features.tolist(), columns=feature_labels, index=df_long.index)
df_full = pd.concat([df_long.drop(columns=["composition"]), features_df], axis=1)
df_full = df_full.dropna(subset=feature_labels)

# --- select exactly the agreed variable set ---
tier1 = [
    "MagpieData mean Electronegativity",
    "MagpieData mean NValence",
    "MagpieData mean CovalentRadius",
    "MagpieData range Electronegativity",
]
tier2 = [
    "MagpieData mean GSbandgap",         # not used -- band gap taken from optb88vdw_bandgap instead
    "MagpieData mean MeltingT",
    "MagpieData mean NUnfilled",
]
tier1 = [c for c in tier1 if c in df_full.columns]
tier2 = [c for c in tier2 if c in df_full.columns and c != "MagpieData mean GSbandgap"]

final_cols = ["jid", "formula", "family", "doping_type", "carrier_sign",
              "optb88vdw_bandgap", "ehull", "Seebeck"] + tier1 + tier2
final_cols = [c for c in final_cols if c in df_full.columns]

df_sisso = df_full[final_cols].dropna().reset_index(drop=True)
print(f"Final SISSO-ready rows: {len(df_sisso)}")
print(f"Input variables: {[c for c in df_sisso.columns if c not in ['jid','formula','family','doping_type','Seebeck']]}")
df_sisso.head()

## 7. Save CSVs

In [ ]:
df_sisso.to_csv("TE_families_SISSO_ready.csv", index=False)
df_sisso[df_sisso["family"] == "bismuth_telluride"].to_csv("bismuth_telluride_SISSO_ready.csv", index=False)
df_sisso[df_sisso["family"] == "half_heusler"].to_csv("half_heusler_SISSO_ready.csv", index=False)

print("Saved: TE_families_SISSO_ready.csv, bismuth_telluride_SISSO_ready.csv, half_heusler_SISSO_ready.csv")

## 8. (Optional) Download

In [ ]:
from google.colab import files
files.download("TE_families_SISSO_ready.csv")
files.download("bismuth_telluride_SISSO_ready.csv")
files.download("half_heusler_SISSO_ready.csv")